In [8]:
import json
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
from PIL import Image
from tqdm import tqdm
from transformers import (
    LayoutLMv3Processor,
    LayoutLMv3Model,
)
import pandas as pd
import numpy as np
import os


In [9]:
# ===================================
# CONFIG
# ===================================
class Config:
    BASE_DIR = Path('.')
    TRAIN_DIR = BASE_DIR / "train_valid/train"
    OUTPUT_DIR = BASE_DIR / "output"
    RESIZED_W = 224
    RESIZED_H = 224
    IMAGE_WIDTH = 224
    IMAGE_HEIGHT = 224
    MAX_LEN = 512
    MODEL_NAME = "microsoft/layoutlmv3-base"

    # 데이터 경로
    REPORT_JSON_DIR = TRAIN_DIR / "report_json"
    REPORT_JPG_DIR  = TRAIN_DIR / "report_jpg"
    PRESS_JSON_DIR  = TRAIN_DIR / "press_json"
    PRESS_JPG_DIR   = TRAIN_DIR / "press_jpg"

    # 이미지 크기 (JSON에서 제공됨)
    IMAGE_WIDTH  = 2480
    IMAGE_HEIGHT = 3508

    MODEL_NAME = "microsoft/layoutlmv3-base"
    MAX_LEN = 512
    BATCH_SIZE = 2
    LR = 5e-5
    EPOCHS = 10

    OUTPUT_DIR.mkdir(exist_ok=True)


In [10]:
import json
import pickle
from pathlib import Path
from PIL import Image
from tqdm import tqdm

ORIG_W = 2480
ORIG_H = 3508
NEW_W = 224
NEW_H = 224


def scale_bbox(bbox):
    """bbox = [x, y, w, h] → 새로운 좌표로 스케일링"""
    x, y, w, h = bbox
    
    sx = NEW_W / ORIG_W
    sy = NEW_H / ORIG_H
    
    return [
        x * sx,
        y * sy,
        w * sx,
        h * sy
    ]


def process_from_pkl(pkl_path, resized_img_dir, resized_json_dir):
    pkl_path = Path(pkl_path)
    resized_img_dir = Path(resized_img_dir)
    resized_json_dir = Path(resized_json_dir)

    resized_img_dir.mkdir(parents=True, exist_ok=True)
    resized_json_dir.mkdir(parents=True, exist_ok=True)

    # Load PKL
    with open(pkl_path, "rb") as f:
        data = pickle.load(f)

    # 대상: train + valid (press + report)
    targets = (
        data["train"]["press"]
        + data["train"]["report"]
        + data["valid"]["press"]
        + data["valid"]["report"]
    )

    print(f"Total PKL entries: {len(targets)}")

    for item in tqdm(targets):
        json_path = Path(item["json_path"])
        img_path = Path(item["img_path"])

        # --- 1) 이미지 리사이즈 후 저장 ---
        try:
            img = Image.open(img_path).convert("RGB")
            img = img.resize((NEW_W, NEW_H), Image.Resampling.LANCZOS)
            img_save_path = resized_img_dir / img_path.name
            img.save(img_save_path)
        except:
            print("Image failed:", img_path)
            continue
        
        # --- 2) JSON 로드 및 bbox 변환 ---
        try:
            with open(json_path, "r", encoding="utf-8") as f:
                js = json.load(f)
        except:
            print("JSON failed:", json_path)
            continue
        
        annos = js.get("learning_data_info", {}).get("annotation", [])
        for a in annos:
            if "bounding_box" in a:
                a["bounding_box"] = scale_bbox(a["bounding_box"])
        
        # 저장
        json_save_path = resized_json_dir / json_path.name
        with open(json_save_path, "w", encoding="utf-8") as f:
            json.dump(js, f, ensure_ascii=False, indent=2)

    print("Done!")


In [ ]:
from torch.utils.data import Dataset
from pathlib import Path
from PIL import Image
import json
import torch
from tqdm import tqdm

NEW_W = 224
NEW_H = 224


class DocumentVQADataset(Dataset):
    def __init__(self, pkl_entries, resized_json_dir, resized_img_dir, processor, is_train=True):
        self.entries = pkl_entries
        self.json_dir = Path(resized_json_dir)
        self.img_dir = Path(resized_img_dir)
        self.processor = processor
        self.is_train = is_train

        self.samples = self._load_samples()

    def _load_samples(self):
        samples = []

        for item in self.entries:
            json_name = Path(item["json_path"]).name
            img_name  = Path(item["img_path"]).name

            json_file = self.json_dir / json_name
            img_file  = self.img_dir / img_name

            if not json_file.exists() or not img_file.exists():
                continue

            try:
                with open(json_file, "r", encoding="utf-8") as f:
                    data = json.load(f)
            except:
                continue

            annos = data["learning_data_info"]["annotation"]
            vc    = data["learning_data_info"].get("visual_context", "")

            for anno in annos:
                # V01 = 표/차트 + 질문이 있는 Annotation
                if anno.get("class_id") == "V01":
                    question = anno.get("visual_instruction", "")
                    bbox     = anno.get("bounding_box", [])
                    iid      = anno["instance_id"]

                    if question and len(bbox) == 4:
                        samples.append({
                            "json": json_file,
                            "image": img_file,
                            "bbox": bbox,
                            "question": question,
                            "instance_id": iid,
                            "context": vc
                        })

        return samples

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        item = self.samples[idx]

        # -----------------------------------------------------
        # 1) Load image (grayscale → RGB)
        # -----------------------------------------------------
        image = Image.open(item["image"]).convert("L").convert("RGB")

        # -----------------------------------------------------
        # 2) Question Tokens
        # -----------------------------------------------------
        q_tokens = item["question"].split()
        q_bboxes = [[0,0,1000,1000]] * len(q_tokens)

        # -----------------------------------------------------
        # 3) Document Context Tokens
        # -----------------------------------------------------
        doc_tokens = item["context"].split()
        doc_bboxes = [[0,0,1000,1000]] * len(doc_tokens)

        # -----------------------------------------------------
        # 4) Merge tokens
        # -----------------------------------------------------
        tokens = q_tokens + doc_tokens
        token_boxes = q_bboxes + doc_bboxes

        # -----------------------------------------------------
        # 5) Processor Encoding
        # -----------------------------------------------------
        enc = self.processor(
            image,
            text=tokens,
            boxes=token_boxes,
            padding="max_length",
            truncation=True,
            max_length=Config.MAX_LEN,
            return_tensors="pt"
        )
        enc = {k: v.squeeze(0) for k, v in enc.items()}

        # -----------------------------------------------------
        # 6) Normalized Label BBox (224×224 기준)
        # -----------------------------------------------------
        if self.is_train:
            x, y, w, h = item["bbox"]
            enc["labels"] = torch.tensor([
                x / NEW_W,
                y / NEW_H,
                w / NEW_W,
                h / NEW_H
            ], dtype=torch.float32)

        enc["instance_id"] = item["instance_id"]
        enc["question_text"] = item["question"]

        return enc


In [12]:
# ======================================================
# MODEL
# ======================================================
class LayoutLMv3ForBBox(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = LayoutLMv3Model.from_pretrained(Config.MODEL_NAME)
        self.reg_head = nn.Linear(self.encoder.config.hidden_size, 4)

    def forward(self, input_ids, bbox, pixel_values, attention_mask, labels=None):
        outputs = self.encoder(
            input_ids=input_ids,
            bbox=bbox,
            pixel_values=pixel_values,
            attention_mask=attention_mask
        )

        cls = outputs.last_hidden_state[:, 0, :]
        pred = self.reg_head(cls)

        loss = None
        if labels is not None:
            loss = nn.MSELoss()(pred, labels)

        return pred, loss


In [13]:
# ======================================================
# TRAIN LOOP
# ======================================================
def train_fn(model, loader, optimizer, device):
    model.train()
    total = 0

    for batch in tqdm(loader, desc="Training"):
        optimizer.zero_grad()

        batch = {k: v.to(device) for k,v in batch.items() if k!="instance_id" and k!="question"}

        preds, loss = model(**batch)

        loss.backward()
        optimizer.step()
        total += loss.item()

    return total / len(loader)


In [14]:
def predict_fn(model, loader, device):
    model.eval()
    preds = []

    with torch.no_grad():
        for batch in tqdm(loader, desc="Predicting"):
            instance_ids = batch["instance_id"]
            questions = batch["question"]

            batch = {k:v.to(device) for k,v in batch.items() 
                     if k not in ["instance_id","question"]}

            pred, _ = model(**batch)
            pred = pred.cpu().numpy()

            for i, pid in enumerate(instance_ids):
                x = pred[i,0] * Config.IMAGE_WIDTH
                y = pred[i,1] * Config.IMAGE_HEIGHT
                w = pred[i,2] * Config.IMAGE_WIDTH
                h = pred[i,3] * Config.IMAGE_HEIGHT

                preds.append({
                    "query_id": pid,
                    "query_text": questions[i],
                    "pred_x": x,
                    "pred_y": y,
                    "pred_w": w,
                    "pred_h": h
                })

    return preds


In [ ]:
# ======================================================
# MAIN TRAINING
# ======================================================
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# 1) Processor (OCR OFF)
processor = LayoutLMv3Processor.from_pretrained(
    Config.MODEL_NAME,
    apply_ocr=False
)

# 2) Load PKL entries
with open("output/sample_index_20percent.pkl", "rb") as f:
    pkl = pickle.load(f)

train_entries = pkl["train"]["report"] + pkl["train"]["press"]
valid_entries = pkl["valid"]["report"] + pkl["valid"]["press"]

print(f"Train entries: {len(train_entries)}")
print(f"Valid entries: {len(valid_entries)}")

# 3) Dataset
train_ds = DocumentVQADataset(
    pkl_entries=train_entries,
    resized_json_dir="resized_json",
    resized_img_dir="resized_img",
    processor=processor,
    is_train=True
)

print("Dataset Loaded:", len(train_ds), "train samples")

# 4) DataLoader (속도 최적화 포함)
train_loader = DataLoader(
    train_ds,
    batch_size=Config.BATCH_SIZE,
    shuffle=True,
    num_workers=4,       # <<< 속도 핵심 (CPU 전처리 병렬화)
    pin_memory=True      # <<< GPU 전송 최적화
)

# 5) Model
model = LayoutLMv3ForBBox().to(device)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=Config.LR
)

# 6) Training Loop
for epoch in range(1, Config.EPOCHS + 1):
    print(f"\n===== Epoch {epoch} =====")
    
    loss = train_fn(model, train_loader, optimizer, device)
    
    print(f"[Epoch {epoch}] Train Loss = {loss:.4f}")

    # Optional save
    torch.save(
        model.state_dict(),
        Config.OUTPUT_DIR / f"model_epoch_{epoch}.pt"
    )

# 7) Final Save
torch.save(
    model.state_dict(),
    Config.OUTPUT_DIR / "model_final.pt"
)

print("\nTraining finished successfully!")


ValueError: num_samples should be a positive integer value, but got num_samples=0

In [ ]:
# ======================================================
# TEST SET LOADING
# ======================================================
test_jsons = list((Config.TRAIN_DIR / "test_json").glob("*.json"))
test_jpg_root = Config.TRAIN_DIR / "test_jpg"

test_ds = DocumentVQADataset(
    json_files=test_jsons,
    jpg_root=test_jpg_root,
    processor=processor,
    is_train=False
)

test_loader = DataLoader(test_ds, batch_size=Config.BATCH_SIZE)

# ======================================================
# PREDICT
# ======================================================
model.load_state_dict(torch.load(Config.OUTPUT_DIR / "model.pt"))
results = predict_fn(model, test_loader, device)

df = pd.DataFrame(results)
df = df[["query_id","query_text","pred_x","pred_y","pred_w","pred_h"]]
df.to_csv(Config.OUTPUT_DIR / "submission.csv", index=False)
df.head()
